# DATA 606 Capstone Deliverable: Comprehensive EDA

**Author:** Brett Allen (ballen3@umbc.edu)

**Topic:** Air Traffic Controller (ATC) Communications Intelligence

*Speech Recognition and Information Extraction for Aviation Safety*

## Setup

In [1]:
!python --version

Python 3.12.9


In [2]:
!cat ../requirements.txt

# Python 3.12.4
datasets==3.2.0
transformers==4.57.6
librosa==1.0.0
soundfile==0.14.0
coverage>=7.7.0  # numba's coverage integration needs coverage.types.Tracer (added in 7.7.0)
pandas==2.3.3
numpy==2.5.3
matplotlib==3.10.6
seaborn==0.13.2



In [3]:
!python -m pip install -q -r ../requirements.txt

### Imports

In [4]:
from datasets import load_dataset, DatasetDict, disable_progress_bars
import os
import io
import base64
import html as html_lib
import sys
from IPython.display import HTML, display
import soundfile as sf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

# Make the reusable pipeline code in ../src importable from the notebook.
sys.path.append(os.path.join(os.getcwd(), "..", "src"))
from data_ingest_pipeline import DataIngestPipeline

### Configurations

In [5]:
%matplotlib inline
sns.set_theme(style="whitegrid")
disable_progress_bars()  # keep re-runs free of noisy download/map/cast tqdm output

# Saving EDA visualizations locally for further analysis.
FIGURES_DIR = os.path.join(os.getcwd(), "..", "res", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)


def save_fig(fig, name, dpi=150):
    """Save a matplotlib figure to res/figures/<name>.png for reuse in slides."""
    path = os.path.join(FIGURES_DIR, f"{name}.png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved figure: {os.path.relpath(path, os.getcwd())}")
    return path


# Saving audio samples locally for further analysis.
AUDIO_SAMPLES_DIR = os.path.join(os.getcwd(), "..", "res", "audio_samples")
os.makedirs(AUDIO_SAMPLES_DIR, exist_ok=True)

In [6]:
data_dir = "../data/"

## Data Ingestion
As part of the proposal EDA process, we looked at two different datasets. The first dataset is the ATCO2-ASR which contains real ATC communications audio. The second dataset is the ATCOSIM which contains simulated data. Both datasets were obtained from Huggingface and joined together along with metadata to identify origin source. Since the ATCO2-ASR dataset represents real ATC communications, it has the possibility of having distortion and noise while the ATCOSIM dataset is always clear and easier to comprehend with no such noise. However, both datasets have speakers with accents which may present as bias when validating the trained model(s) against ATC audio without such accents. 

In this comprehensive EDA process, we introduce the ATC-ASR-Dataset as a third dataset to supplement the real ATCO2-ASR dataset (see https://huggingface.co/datasets/jacktol/ATC-ASR-Dataset). According to the publisher of this dataset, Jacktol: "ATC ASR Dataset is a high-quality, fine-tuning-ready speech recognition dataset constructed from two real-world Air Traffic Control (ATC) corpora: the UWB ATC Corpus and the ATCO2 1-Hour Test Subset." Both datasets feature real ATC utterences just like the ATCO2-ASR dataset.

In [7]:
atco2_asr_path = os.path.join(data_dir, "raw/atco2-asr")
atcosim_path = os.path.join(data_dir, "raw/atcosim")
atc_asr_dataset_path = os.path.join(data_dir, "raw/atc-asr-dataset")

print(f"atco2_asr_path       : {atco2_asr_path}")
print(f"atcosim_path         : {atcosim_path}")
print(f"atc_asr_dataset_path : {atc_asr_dataset_path}")

atco2_asr_path       : ../data/raw/atco2-asr
atcosim_path         : ../data/raw/atcosim
atc_asr_dataset_path : ../data/raw/atc-asr-dataset


### ATCO2-ASR Dataset

In [8]:
atco2_asr_dataset = load_dataset("jlvdoorn/atco2-asr", cache_dir=atco2_asr_path)

In [9]:
atco2_asr_dataset.keys()

dict_keys(['train', 'validation'])

In [10]:
n_train = len(atco2_asr_dataset['train'])
n_val = len(atco2_asr_dataset['validation'])
total = n_train + n_val
print(f"Number of training samples   : {n_train} ({n_train/total:.2%})")
print(f"Number of validation samples : {n_val} ({n_val/total:.2%})")
print(f"Total number of samples      : {total}")
print(f"Split ratio (train:val)      : {n_train/n_val:.2f}")

Number of training samples   : 446 (79.79%)
Number of validation samples : 113 (20.21%)
Total number of samples      : 559
Split ratio (train:val)      : 3.95


In [11]:
# Inspect the first training sample to see what the data looks like.
atco2_asr_dataset['train'][0]

{'audio': {'path': 'LKPR_RUZYNE_Radar_120_520MHz_20201025_091112.wav',
  'array': array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         -6.10351562e-05, -6.10351562e-05, -6.10351562e-05], shape=(117760,)),
  'sampling_rate': 16000},
 'text': 'Oscar Kilo Papa Mike Bravo descend flight level one hundred level one hundred Oscar Kilo Papa Mike Bravo ',
 'info': 'LKPR\nPraha Ruzyne\nRadar\nAKEVA ARVEG BAGRU BAROX BAVIN BEKVI ELMEK ELPON ERASU EVEMI KENOK KUVIX LETNA RATEV RISUK SOMIS SULOV TIPRU UTORO\nBLA131 BLA1XQ BTI7PY CTN480 DLH3NL DLH9TP ETD72E EWG6HP FIN1DH IRA711 KLM44K MLD863 MLD864 OKHBT OKLLZ OKMHZ OKPHM OKWUS17 OKYAI14 RYR1JU RYR4945 SXS7D THY32B THY6577 TIE790J UAE73  \nAll Charter Air Baltic Croatia Lufthansa Etihad Eurowings Finn Iranair Klm Moldova Oklahoma Okapi Alfa Ryan Sunexpress Turkish Time Emirates'}

In [12]:
# Get a unique count of sample rates to verify that sample rate is always 16kHz for atco2-asr dataset
# Iterate through the dataset and get the sample rate for each audio file
unique_sample_rates = set()
for split in atco2_asr_dataset.keys():
    for sample in atco2_asr_dataset[split]:
        unique_sample_rates.add(sample['audio']['sampling_rate'])

print(f"Unique sample rates in atco2_asr_dataset: {unique_sample_rates}")

Unique sample rates in atco2_asr_dataset: {16000}


### ATCOSIM Dataset

In [13]:
atcosim_dataset = load_dataset("jlvdoorn/atcosim", cache_dir=atcosim_path)

In [14]:
atcosim_dataset.keys()

dict_keys(['train', 'validation'])

In [15]:
n_train = len(atcosim_dataset['train'])
n_val = len(atcosim_dataset['validation'])
total = n_train + n_val
print(f"Number of training samples   : {n_train} ({n_train/total:.2%})")
print(f"Number of validation samples : {n_val} ({n_val/total:.2%})")
print(f"Total number of samples      : {total}")
print(f"Split ratio (train:val)      : {n_train/n_val:.2f}")

Number of training samples   : 7646 (79.99%)
Number of validation samples : 1913 (20.01%)
Total number of samples      : 9559
Split ratio (train:val)      : 4.00


In [16]:
# Inspect the first training sample to see what the data looks like.
atcosim_dataset['train'][0]

{'audio': {'path': 'gf1_01_001.wav',
  'array': array([ 6.10351562e-05, -3.05175781e-05,  3.05175781e-05, ...,
         -1.95922852e-02, -6.31713867e-03, -9.76562500e-04], shape=(93955,)),
  'sampling_rate': 32000},
 'text': ' contact geneva one two eight decimal one five good bye '}

In [17]:
# Get a unique count of sample rates to verify that sample rate is always 32kHz for atcosim dataset
# Iterate through the dataset and get the sample rate for each audio file
unique_sample_rates = set()
for split in atcosim_dataset.keys():
    for sample in atcosim_dataset[split]:
        unique_sample_rates.add(sample['audio']['sampling_rate'])

print(f"Unique sample rates in atcosim_dataset: {unique_sample_rates}")

Unique sample rates in atcosim_dataset: {32000}


### ATC-ASR Dataset

In [18]:
atc_asr_dataset = load_dataset("jacktol/ATC-ASR-Dataset", cache_dir=atc_asr_dataset_path)

In [19]:
atc_asr_dataset.keys()

dict_keys(['train', 'validation', 'test'])

In [20]:
n_train = len(atc_asr_dataset['train'])
n_val = len(atc_asr_dataset['validation'])
n_test = len(atc_asr_dataset['test'])
total = n_train + n_val + n_test
print(f"Number of training samples   : {n_train} ({n_train/total:.2%})")
print(f"Number of validation samples : {n_val} ({n_val/total:.2%})")
print(f"Number of test samples       : {n_test} ({n_test/total:.2%})")
print(f"Total number of samples      : {total}")

Number of training samples   : 6497 (79.99%)
Number of validation samples : 812 (10.00%)
Number of test samples       : 813 (10.01%)
Total number of samples      : 8122


In [21]:
# Inspect the first training sample to see what the data looks like.
atc_asr_dataset['train'][0]

{'id': '0023HAQRADETCJDF66RO',
 'audio': {'path': '0023HAQRADETCJDF66RO.wav',
  'array': array([-0.00024414, -0.00061035, -0.00112915, ...,  0.00018311,
          0.00030518,  0.00021362], shape=(22720,)),
  'sampling_rate': 16000},
 'text': 'SIERRA DELTA MIKE'}

In [22]:
# Get a unique count of sample rates to verify that sample rate is always 16kHz for atc asr dataset
# Iterate through the dataset and get the sample rate for each audio file
unique_sample_rates = set()
for split in atc_asr_dataset.keys():
    for sample in atc_asr_dataset[split]:
        unique_sample_rates.add(sample['audio']['sampling_rate'])

print(f"Unique sample rates in atc_asr_dataset: {unique_sample_rates}")

Unique sample rates in atc_asr_dataset: {16000}


### Join Datasets
Need to tag the datasets with the source and whether they originally came from training or validation sets, resample ATCOSIM's audio (32kHz) down to ATCO2-ASR's and ATC-ASR's rate (16kHz) so all sources are directly comparable/usable by the same model (e.g., OpenAI Whisper model requires 16kHz sample rate), and derive per-utterance metadata (`duration_sec`, `word_count`, `character_count`, etc.). This is implemented as a reusable `DataIngestPipeline` class rather than inline here so model-training code can run the exact same ingestion later.

In [23]:
# Loading, tagging, schema-alignment, resampling, and metadata derivation all
# live in `DataIngestPipeline` (../src/data_ingest_pipeline.py) so the
# same logic can be reused later during model training (e.g., as a pipline).
#
# `run()` caches the output under `../data/processed/`: the first call does
# the full download + resample (supports running on parallel threads to improve runtime speed; pass
# `num_proc=` to the constructor to do this).
# Later calls just reload from disk. Pass `force=True` to force a full recompute (e.g. after
# changing `target_sample_rate`).
pipeline = DataIngestPipeline(data_dir=data_dir)
combined_dataset, utterance_df = pipeline.run(force=True)
combined_dataset

AssertionError: Combined row count (17427) does not match the sum of source dataset row counts (18240); a row was lost or duplicated during the join.

In [ ]:
# Confirm that the sample rate is now 16kHz for all audio files in the combined dataset
unique_sample_rates = set()
for split in combined_dataset.keys():
    for sample in combined_dataset[split]:
        unique_sample_rates.add(sample['audio']['sampling_rate'])

print(f"Unique sample rates in combined_dataset: {unique_sample_rates}")

`combined_dataset` (above) is the training-ready `DatasetDict`, audio resampled to 16kHz, with `text`/`info`/`audio` plus the new metadata columns. `utterance_df` is a flat, audio-free `pandas.DataFrame` version of the same data (one row per utterance, across both splits) for EDA.

In [ ]:
print(f"utterance_df shape: {utterance_df.shape}")
utterance_df.head()